# ਪਾਠ 02 - ਮਾਇਕ੍ਰੋਸਾਫਟ ਏਜੰਟ ਫ੍ਰੇਮਵਰਕ ਦੀ ਖੋਜ

**ਮਾਇਕ੍ਰੋਸਾਫਟ ਏਜੰਟ ਫ੍ਰੇਮਵਰਕ (MAF)** ਇੱਕ ਏਕਜੁੱਟ ਫ੍ਰੇਮਵਰਕ ਹੈ ਜੋ AI ਏਜੰਟ ਬਣਾਉਣ ਲਈ ਬਣਾਇਆ ਗਿਆ ਹੈ। ਇਹ ਸਾਫ਼, ਸੰਰਚਿਤ ਆਰਕੀਟੈਕਚਰ ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ ਜਿਸ ਵਿੱਚ ਚਾਰ ਮੁੱਖ ਭਾਗ ਹਨ:

- **ਕਲਾਇੰਟ** – ਇੱਕ AI ਮਾਡਲ ਐਂਡਪੌਇੰਟ ਨਾਲ ਜੁੜਦਾ ਹੈ ਅਤੇ ਸੰਚਾਰ ਨੂੰ ਸੰਭਾਲਦਾ ਹੈ
- **ਏਜੰਟ** – ਹਦਾਇਤਾਂ ਅਤੇ ਸੰਦ ਪਰਿਭਾਸ਼ਾਵਾਂ ਦੇ ਨਾਲ ਇੱਕ ਕਲਾਇੰਟ ਨੂੰ ਲਪੇਟਦਾ ਹੈ
- **ਸੰਦ** – ਖਾਸ ਫੰਕਸ਼ਨਾਂ ਨਾਲ ਏਜੰਟ ਦੀ ਸਮਰੱਥਾ ਵਧਾਉਂਦੇ ਹਨ ਜੋ ਮਾਡਲ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ
- **ਸੈਸ਼ਨ** – ਬਹੁ-ਮੁੜ ਕੇ ਗੱਲਬਾਤ ਲਈ ਸੰਵਾਦ ਦਾ ਇਤਿਹਾਸ ਸੰਭਾਲਦਾ ਹੈ

ਇਸ ਪਾਠ ਵਿੱਚ, ਅਸੀਂ ਇੱਕ **ਟ੍ਰੈਵਲ ਬੁਕਿੰਗ ਏਜੰਟ** ਬਣਾਵਾਂਗੇ ਜੋ ਇਨ ਧਾਰਨਾਵਾਂ ਦੀ ਵਰਤੋਂ ਕਰਦਿਆਂ ਮੰਜ਼ਿਲ ਦੀ ਉਪਲਬਧਤਾ ਦੀ ਜਾਂਚ ਕਰਦਾ ਹੈ।


## ਸੈਟਅਪ


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## ਏਜੰਟ ਫਰੇਮਵਰਕ ਆਰਕੀਟੈਕਚਰ ਨੂੰ ਸਮਝਣਾ

ਮਾਈਕ੍ਰੋਸੌਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਇੱਕ ਲੇਅਰਡ ਆਰਕੀਟੈਕਚਰ ਦੀ ਪਾਲਣਾ ਕਰਦਾ ਹੈ:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ਕਲਾਇੰਟ** – ਇੱਕ `FoundryChatClient` ਏਜ਼ੂਅਰ ਓਪਨਏਆਈ ਡਿਪਲੌਇਮੈਂਟ ਨਾਲ ਜੁੜਦਾ ਹੈ। ਇਹ ਪ੍ਰਮਾਣਿਕਤਾ, ਬੇਨਤੀ ਦੇ ਫਾਰਮੈਟਿੰਗ ਅਤੇ ਜਵਾਬ ਦੀ ਵਿਅਖਿਆ ਸੰਭਾਲਦਾ ਹੈ।
2. **ਏਜੰਟ** – ਕਲਾਇੰਟ ਤੋਂ `provider.create_agent()` ਰਾਹੀਂ ਬਣਾਇਆ ਗਿਆ, ਏਜੰਟ ਮਾਡਲ ਐਕਸੈਸ ਨੂੰ ਨਿਰਦੇਸ਼ਾਂ (ਸਿਸਟਮ ਪ੍ਰੋਂਪਟ) ਅਤੇ ਟੂਲਜ਼ ਨਾਲ ਜੋੜਦਾ ਹੈ।
3. **ਟੂਲਜ਼** – ਪਾਇਥਨ ਫੰਕਸ਼ਨ ਜੋ `@tool` ਨਾਲ ਸਜਾਏ ਗਏ ਹਨ, ਜਿਨ੍ਹਾਂ ਨੂੰ ਏਜੰਟ ਕਿਰਿਆਵਾਂ ਕਰਨ ਜਾਂ ਡੇਟਾ ਪ੍ਰਾਪਤ ਕਰਨ ਲਈ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।
4. **ਸੈਸ਼ਨ** – ਇੱਕ `AgentSession` ਵਸਤੂ (ਜੋ `agent.create_session()` ਰਾਹੀਂ ਬਣਾਇਆ ਗਿਆ) ਜੋ ਗਲਬਾਤ ਇਤਿਹਾਸ ਸਟੋਰ ਕਰਦਾ ਹੈ, ਜਿਸ ਨਾਲ ਏਜੰਟ ਪਹਿਲਾਂ ਦੇ ਸੰਦਰਭ ਨੂੰ ਯਾਦ ਰੱਖ ਕੇ ਬਹੁ-ਪਾਸਾ ਸੰਵਾਦ ਸੰਭਵ ਹੁੰਦਾ ਹੈ।

ਚਲੋ ਹਰ ਲੇਅਰ ਨੂੰ ਕਦਮ ਬਦ ਕਦਮ ਬਣਾਈਏ।


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool ਡੈਕੋਰੇਟਰ ਨਾਲ ਟੂਲ ਸ਼ਾਮਲ ਕਰਨਾ

ਟੂਲ ਏਜੰਟ ਨੂੰ ਲਿਖਤ ਤਿਆਰ ਕਰਨ ਤੋਂ ਇਲਾਵਾ ਕਾਰਵਾਈ ਕਰਨ ਦੀ ਆਗਿਆ ਦਿੰਦੇ ਹਨ। `@tool` ਡੈਕੋਰੇਟਰ ਇੱਕ ਆਮ ਪਾਇਥਨ ਫੰਕਸ਼ਨ ਨੂੰ ਕੁਝ ਐਸਾ ਬਣਾ ਦਿੰਦਾ ਹੈ ਜਿਸਨੂੰ ਏਜੰਟ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।

ਮੁੱਖ ਬਿੰਦੂ:
- ਮਾਡਲ ਹਰ ਪੈਰਾਮੀਟਰ ਨੂੰ ਸਮਝ ਸਕੇ, ਇਸ ਲਈ `Annotated[type, "description"]` ਦੀ ਵਰਤੋਂ ਕਰੋ।
- ਡੌਕਸਟ੍ਰਿੰਗ ਟੂਲ ਦਾ ਵੇਰਵਾ ਬਣ ਜਾਂਦਾ ਹੈ ਜੋ ਮਾਡਲ ਵੇਖਦਾ ਹੈ।
- `approval_mode="never_require"` ਦਾ ਮਤਲਬ ਹੈ ਕਿ ਟੂਲ ਯੂਜ਼ਰ ਦੀ ਪੁਸ਼ਟੀ ਦੇ ਬਿਨਾਂ ਖੁਦ-ਬ-ਖੁਦ ਚੱਲਦਾ ਹੈ।


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ਟੂਲਸ ਨਾਲ ਏਜੰਟ ਬਣਾਉਣਾ

ਹੁਣ ਅਸੀਂ ਕਲਾਇੰਟ, ਹੁੱਕਮਾਂ ਅਤੇ ਟੂਲਸ ਨੂੰ ਇੱਕ ਏਜੰਟ ਵਿੱਚ ਮਿਲਾਉਂਦੇ ਹਾਂ। `instructions` ਸਿਸਟਮ ਪ੍ਰੰਪਟ ਵਜੋਂ ਕੰਮ ਕਰਦੇ ਹਨ — ਇਹ ਏਜੰਟ ਦੀ ਸ਼ਖਸੀਅਤ ਅਤੇ ਵਿਹਾਰ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਨ।


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## ਸੈਸ਼ਨਾਂ ਨਾਲ ਬਹੁ-ਮੋੜੀ ਗੱਲਬਾਤ

ਇੱਕ `AgentSession` (ਜੋ `agent.create_session()` ਦੇ ਜ਼ਰੀਏ ਬਣਾਇਆ ਗਿਆ ਹੈ) ਗੱਲਬਾਤ ਵਿੱਚ ਸਾਰੇ ਸੁਨੇਹਿਆਂ ਦੀ ਟਰੈਕਿੰਗ ਕਰਦਾ ਹੈ। ਹਰ `agent.run()` ਕਾਲ ਵਿੱਚ ਇੱਕੋ ਸੈਸ਼ਨ ਦੇਣ ਨਾਲ, ਏਜੰਟ ਪੂਰੇ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਤੱਕ ਪਹੁੰਚ ਰੱਖਦਾ ਹੈ ਅਤੇ ਪੁਰਾਣੇ ਸੁਨੇਹਿਆਂ ਨੂੰ ਦੇਖ ਸਕਦਾ ਹੈ।

ਅਸੀਂ `tools=[check_destination_availability]` ਦੇ ਰਹੇ ਹਾਂ ਤਾਂ ਜੋ ਏਜੰਟ ਹਰ ਮੋੜ ਵਿੱਚ ਸਾਡਾ ਉਪਲੱਬਧਤਾ ਚੈੱਕ ਕਰਨ ਵਾਲਾ ਚੈੱਕਰ ਕਾਲ ਕਰ ਸਕੇ।


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## ਸਾਰ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਮਾਈਕ੍ਰੋਸੌਫਟ ਏਜੰਟ ਫਰੇਮਵર્ક ਦੇ ਚਾਰ ਸਤਰਾਂ ਦੀ ਖੋਜ ਕੀਤੀ:

| ਸੰਕਲਪ | ਤੁਸੀਂ ਕੀ ਸਿੱਖਿਆ |
|---------|------------------|
| **ਕਲਾਇੰਟ** | `FoundryChatClient` ਕ੍ਰੈਡੈਂਸ਼ਲ-ਆਧਾਰਿਤ ਆਥ ਨਾਲ Azure OpenAI ਨਾਲ ਜੁੜਦਾ ਹੈ |
| **ਏਜੰਟ** | `provider.create_agent()` ਮਾਡਲ ਕਨੈਕਸ਼ਨ ਨੂੰ ਦਿਸ਼ਾ-ਨਿਰਦੇਸ਼ਾਂ ਅਤੇ ਨਾਮ ਨਾਲ ਬੰਨ੍ਹਦਾ ਹੈ |
| **ਟੂਲਸ** | `@tool` ਡੈਕੋਰੇਟਰ ਏਜੰਟ ਲਈ ਪਾਇਥਨ ਫੰਕਸ਼ਨਾਂ ਨੂੰ ਕਾਲ ਕਰਨ ਲਈ ਪ੍ਰਗਟ ਕਰਦਾ ਹੈ |
| **ਸੈਸ਼ਨ** | `agent.create_session()` ਕਈ ਦੌਰਾਂ ਵਿੱਚ ਗੱਲਬਾਤ ਦਾ ਇਤਿਹਾਸ ਸੰਭਾਲਦਾ ਹੈ |

ਇਹ ਨਿਰਮਾਣ ਖੰਡ ਇਕੱਠੇ ਹੋ ਕੇ ਐਜੰਟ ਬਣਾਉਂਦੇ ਹਨ ਜੋ ਕੁਦਰਤੀ ਗੱਲਬਾਤ ਕਰ ਸਕਦੇ ਹਨ, ਬਾਹਰੀ ਫੰਕਸ਼ਨਾਂ ਨੂੰ ਕਾਲ ਕਰ ਸਕਦੇ ਹਨ, ਅਤੇ ਸੰਦਰਭ ਬਰਕਰਾਰ ਰੱਖ ਸਕਦੇ ਹਨ — ਜੋ ਕਿ ਅਗਲੇ ਪਾਠਾਂ ਵਿੱਚ ਵਧੇਰੇ ਉਦਯਮਸ਼ੀਲ ਪੈਟਰਨਾਂ ਲਈ ਬੁਨਿਆਦ ਹੈ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
